# New pricing page: conversion against the old 13%

**One-Sample Test for a Proportion** · a worked example from the StatNex teaching guide

**The story.** An app used to convert 13% of free-trial users into paying customers. After launching a new pricing page, 78 of the next 500 trial users paid, a 15.6% conversion rate. The product team wants to celebrate.

**The question.** Is the new page's conversion rate really different from the old 13%?

**To run it**, choose **Runtime → Run all**. Colab first warns that this notebook was not written by Google; it reads none of your data, so choose **Run anyway**. The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `02_one_sample/one_sample_proportion_test.csv` (2 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
Variable,New pricing page
Sample size,500.0
Conversion rate,0.156

## Libraries

In [ ]:
import numpy as np
import pandas as pd
from scipy import stats

pd.set_option("display.width", 120)
pd.set_option("display.float_format", "{:.6g}".format)


def stars(p):
    """* for P < 0.05, ** for P < 0.01."""
    return "**" if p < 0.01 else "*" if p < 0.05 else ""

## The data

In [ ]:
# Each column holds the sample size and the proportion, top to bottom.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
variables = ["New pricing page"]
# The population proportion under the null hypothesis.
p0 = 0.13

## Exact binomial test, written with the F distribution

In [ ]:
# If the true rate were p0, how often would chance give a count at least as far from
# n p0 as the observed k? That is the one-sided P: the binomial tail on the side the
# data lie, P(X <= k) when k is at or below n p0, P(X >= k) above it. SciPy's exact
# binomial test gives it. The app prints the same tail in its F form, so F and its two
# degrees of freedom are worked out beside it. The test assumes independent trials
# with one common rate, and a benchmark p0 that is a fixed number, not an estimate.
results = {}
for name in variables:
    size, p = pd.to_numeric(data[name], errors="coerce").dropna().to_numpy(float)[:2]
    n = int(round(size))
    k = int(round(n * p))
    if k <= n * p0:
        df1, df2 = 2 * (k + 1), 2 * (n - k)
        f = (n - k) / (k + 1) * p0 / (1 - p0)
    else:
        df1, df2 = 2 * (n - k + 1), 2 * k
        f = k / (n - k + 1) * (1 - p0) / p0
    side = "less" if k <= n * p0 else "greater"
    p_one = stats.binomtest(k, n, p0, alternative=side).pvalue
    # The app's two-sided P doubles the one-sided one (the convention that matches the
    # Clopper-Pearson interval). stats.binomtest(k, n, p0).pvalue instead adds up every
    # outcome no more likely than k, as R's binom.test and Stata's bitest do, and is
    # usually a little smaller.
    p_two = min(1.0, 2 * p_one)
    results[name] = {"n": n, "proportion": p, "p0": p0, "difference": abs(p - p0),
                     "F": f, "df1": df1, "df2": df2,
                     "P two-sided": p_two, "P one-sided": p_one}

table = pd.DataFrame(results)
print(table)
print()
print(table.loc[["P two-sided", "P one-sided"]].map(stars))

## What the result says

Of the 500 trial users who saw the new page, a proportion of 0.156 paid, against the old 0.13. The difference is 0.026, or 2.6 percentage points.

The test asks if chance alone could give such a difference. It uses an exact method based on the F distribution, with F = 1.23. The two-sided P value is 0.102. A P value is the chance of a difference at least this large if the true rate were still 13%. A chance of about one in ten is not small, so the result is not significant.

The answer is: not yet. A real gain is not shown, but it is not ruled out either. And a comparison with an old rate could not prove that the page caused a gain anyway.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.